# 03: Trustworthiness

*How well-evidenced and traceable is the KG content?*

This notebook evaluates **trustworthiness** across all KGs listed in `config.yaml`, examining whether the content of each KG is grounded in diverse, traceable, and well-characterised evidence.

| Sub-metric | Definition | Scoring |
|------------|-----------|---------|
| **Edge traceability** | Resolution at which edges can be traced to their origin database | Ordinal tier → 0-1 mapping: Tier 0 (type-level) = 0.0, Tier 1 (per-node provenance) = 0.5, Tier 2 (per-edge provenance) = 1.0 |
| **Source diversity** | Breadth of contributing databases | Log-normalised: log(n_sources) / log(max_sources across KGs), yielding 0-1 score |
| **Uncertainty Quantification** | Fraction of edges with confidence/evidence-strength annotation | Direct ratio: edges_with_score / total_edges (0-1) |

**Scoring.** The dimension score is the equal-weighted mean of edge traceability, uncertainty quantification, and source diversity. Source diversity is log-normalised against the maximum observed source count across the benchmark KGs, yielding a 0-1 score, a KG with fewer but higher-quality sources is not necessarily less trustworthy. Edge traceability distinguishes between *node-level* provenance (knowing which database an entity identifier comes from) and *edge-level* provenance (knowing which database asserted the relationship). UQ coverage penalises the absence of per-edge confidence scores, currently 0.0 for all benchmark KGs.

**Inputs:** `config.yaml` · KG edge/node files (via `load_kg`)

**Outputs:** `results/figures/03_source_diversity.{pdf,png}` · `results/trustworthiness/source_granularity.csv` · `results/checkpoints/03_trustworthiness.pkl`

**Dependencies:** `src/loading.py` · `src/plotting.py`

## Set-up

Three knowledge graphs are evaluated: **Hetionet** (Himmelstein et al., 2017), **DRKG** (Ioannidis et al., 2020), and **PrimeKG** (Chandak et al., 2023). All graphs and node tables are loaded using the shared `src/` loading utilities and evaluated under identical conditions. The same random seed (`seed = 42`) and negative sampling strategy (random drug-disease pairs not in the known indication set) are used throughout.


In [1]:
# Imports
import sys, os, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from typing import List

_root = Path(os.path.abspath('')).resolve()
_root = _root.parent if _root.name == 'eval_notebooks' else _root
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.plotting import (setup_style, clean_ax, save_fig, panel_label,
                           TEXT_COLOR, TICK_COLOR, KG_PALETTE)
from src.loading  import find_config, load_config, load_kg

setup_style()

In [2]:
# Config and paths
config  = load_config(find_config(_root))
params  = config['analysis_params']
BASE    = config['_base_dir']
FIGS    = BASE / 'results' / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)

In [3]:
# Free any KG/graph state left over from a previous notebook (prevents
# the ~150 GB OOM that hit when running notebooks back-to-back in one kernel).
for _v in ('kgs', 'graphs', 'maps', 'preps'):
    try:
        del globals()[_v]
    except KeyError:
        pass
import gc
gc.collect()

# Load KGs
kgs = {}
for name in config['knowledge_graphs']:
    try:
        kg, nodes = load_kg(name, config)
        kgs[name] = {'kg': kg, 'nodes': nodes}
        print(f'{name:12s}: {len(kg):,} edges, {len(nodes):,} nodes')
    except FileNotFoundError: print(f'{name:12s}: [NOT FOUND - skipped]')
    except Exception as e:    print(f'{name:12s}: [ERROR: {e}]')

KG_NAMES  = list(kgs.keys())
LOADED    = [n for n in kgs if config['knowledge_graphs'].get(n, {}).get('relations')]
KG_COLORS = {n: KG_PALETTE.get(n, '#888888') for n in KG_NAMES}

primekg     : 8,100,498 edges, 129,375 nodes


hetionet    : 2,250,197 edges, 47,031 nodes


drkg        : 5,874,261 edges, 97,238 nodes


openbilink  : 4,986,521 edges, 185,929 nodes


biokg       : 2,067,998 edges, 105,524 nodes
  matrix: loading from cache _cache_matrix_9229042b_kg.parquet / _cache_matrix_9229042b_nodes.parquet


  matrix: cached load — 2,850,122 nodes, 41,595,079 edges
matrix      : 41,595,079 edges, 2,850,122 nodes


## 1. Source Diversity

Source diversity characterises how many independent databases underpin a KG. Edges corroborated by multiple sources are more likely to reflect genuine biology; edges derived from a single source represent a single point of failure that can propagate curation bias, a well-documented phenomenon whereby biomedical databases disproportionately annotate well-studied entities at the expense of under-investigated ones (Haynes et al. 2018; Cheung et al. 2013).

**Source counting methodology.** We count the number of *distinct primary upstream databases* that contributed edges to each KG, as documented in the original publications:

- **PrimeKG** (Chandak et al. 2023): 20 primary resources. Per-edge `x_source` / `y_source` fields record higher-level source groupings (9 categories), but the KG integrates data from 20 distinct databases.
- **DRKG** (Ioannidis et al. 2020): 7 primary resources. Source encoded in relation name prefix (`SOURCE::rel::Types`).
- **Hetionet** (Himmelstein et al. 2017): 29 primary resources. No per-edge source field in the v1.0 release; sources documented in the original publication.

> Haynes, W.A., Tomczak, A. & Khatri, P. Gene annotation bias impedes biomedical research. *Sci. Rep.* **8**, 1362 (2018).
>
> Cheung, W.A. et al. Compensating for literature annotation bias when predicting novel drug-disease relationships. *BMC Med. Genomics* **6**(Suppl 2), S3 (2013).
>
> Himmelstein, D.S. et al. Systematic integration of biomedical knowledge prioritizes drugs for repurposing. *eLife* **6**, e26726 (2017).

In [4]:
# Relation / source distribution per KG
KG_SCHEMA = {
    'primekg': {
        'mode':  'columns',
        'cols':  ('x_source', 'y_source'),
        'label': 'source',
    },
    'drkg': {
        'mode':  'relation_prefix',
        'label': 'source',
    },
    'hetionet': {
        'mode':   'relation_values',
        'label':  'metaedge',
        'decode': {
            'GpBP': 'Gene-participates-BiologicalProcess',
            'AeG':  'Anatomy-expresses-Gene',
            'Gr>G': 'Gene-regulates-Gene',
            'GiG':  'Gene-interacts-Gene',
            'GrG':  'Gene-regulates-Gene (directed)',
            'CcSE': 'Compound-causes-SideEffect',
            'AdG':  'Anatomy-downregulates-Gene',
            'AuG':  'Anatomy-upregulates-Gene',
            'GpMF': 'Gene-participates-MolecularFunction',
            'GpPW': 'Gene-participates-Pathway',
            'GpCC': 'Gene-participates-CellularComponent',
            'GcG':  'Gene-covaries-Gene',
            'CdG':  'Compound-downregulates-Gene',
            'CuG':  'Compound-upregulates-Gene',
            'DaG':  'Disease-associates-Gene',
            'CtD':  'Compound-treats-Disease',
            'CbG':  'Compound-binds-Gene',
            'DrD':  'Disease-resembles-Disease',
            'DuG':  'Disease-upregulates-Gene',
            'DdG':  'Disease-downregulates-Gene',
            'DpS':  'Disease-presents-Symptom',
            'PCiC': 'PharmacologicClass-includes-Compound',
            'CrC':  'Compound-resembles-Compound',
            'DlA':  'Disease-localizes-Anatomy',
            'CpD':  'Compound-palliates-Disease',
        },
    },
    # OpenBioLink x_source / y_source both carry the asserting database per edge
    # (STRING, STITCH, Bgee, GO, HPO, CDT, DisGeNet, SIDER, UBERON, DO, DrugCentral).
    # Both columns are always identical (set to the edge's source in load_openbilink)
    # so _compute_counts divides by 2 correctly to avoid double-counting.
    'openbilink': {
        'mode':  'columns',
        'cols':  ('x_source', 'y_source'),
        'label': 'source',
    },
    # MATRIX (Every Cure / Biolink) edge_source carries the per-edge
    # asserting database (94 distinct infores CURIEs). x_source / y_source
    # carry per-NODE upstream pipeline (rtxkg2 / robokop / primekg) and are
    # used separately for tier-1 node provenance.
    'matrix': {
        'mode':  'columns',
        'cols':  ('edge_source',),
        'label': 'source',
    },
    'biokg': {
        'mode':   'relation_values',
        'label':  'predicate',
    },
}

def _compute_counts(kg: pd.DataFrame, schema: dict) -> pd.DataFrame:
    '''Compute edge-count distribution for one KG given its schema entry.'''
    mode, label = schema['mode'], schema['label']
    if mode == 'columns':
        cols = schema['cols']
        # Single-col mode (e.g. edge_source for MATRIX) count once.
        # Two-col mode (PrimeKG, OpenBioLink) both x/y_source carry the same
        # value duplicated per edge, so divide by 2 to avoid double-counting.
        denom = len(cols)
        counts = (pd.concat([kg[c] for c in cols])
                  .dropna().value_counts() // denom)
        counts = counts.rename_axis(label).rename('n_edges').reset_index()
    elif mode == 'relation_prefix':
        counts = (kg['relation'].str.extract(r'^([^:]+)::')[0].fillna('other')
                  .value_counts().rename_axis(label).rename('n_edges').reset_index())
    elif mode == 'relation_values':
        counts = (kg['relation'].value_counts()
                  .rename_axis(label).rename('n_edges').reset_index())
        if 'decode' in schema:
            counts['description'] = counts[label].map(schema['decode'])
    return counts

kg_counts = {}
for name, data in kgs.items():
    schema = KG_SCHEMA.get(name)
    if schema is None:
        print(f'{name}: no schema defined - skipped')
        continue
    counts = _compute_counts(data['kg'], schema)
    kg_counts[name] = counts
    print(f'\n{name} - {schema["label"]} distribution '
          f'({len(counts)} distinct, {counts["n_edges"].sum():,} total edges)')
    display(counts.set_index(schema['label']))


primekg - source distribution (9 distinct, 8,100,498 total edges)


,n_edges
source,
DrugBank,2805696
NCBI,2631229
UBERON,1566154
GO,442027
MONDO,268349
HPO,257096
MONDO_grouped,72895
REACTOME,47716
CTD,9336



hetionet - metaedge distribution (24 distinct, 2,250,197 total edges)


,n_edges,description
metaedge,,
GpBP,559504,Gene-participates-BiologicalProcess
AeG,526407,Anatomy-expresses-Gene
Gr>G,265672,Gene-regulates-Gene
GiG,147164,Gene-interacts-Gene
CcSE,138944,Compound-causes-SideEffect
AdG,102240,Anatomy-downregulates-Gene
AuG,97848,Anatomy-upregulates-Gene
GpMF,97222,Gene-participates-MolecularFunction
GpPW,84372,Gene-participates-Pathway



drkg - source distribution (7 distinct, 5,874,261 total edges)


...

,n_edges
source,
STRING,1840625
Bgee,1728138
STITCH,456423
GO,361095
HPO,228975
CDT,135809
DisGeNet,93885
SIDER,89096
UBERON,33344



biokg - predicate distribution (17 distinct, 2,067,998 total edges)


,n_edges
predicate,
DDI,1334085
PROTEIN_PATHWAY_ASSOCIATION,250118
PPI,113817
PROTEIN_DISEASE_ASSOCIATION,109276
MEMBER_OF_COMPLEX,87452
DRUG_DISEASE_ASSOCIATION,66867
DPI,28033
COMPLEX_IN_PATHWAY,21125
COMPLEX_TOP_LEVEL_PATHWAY,15639



matrix - source distribution (81 distinct, 41,595,079 total edges)


,n_edges
source,
infores:string,10852506
infores:semmeddb,7570124
infores:primekg,4789363
infores:hmdb,2303592
infores:drugbank,2273040
...,...
infores:hl7-umls,98
infores:ncbi-taxonomy,12
infores:ino,7


In [5]:
# Source database diversity across KGs
PANEL_CONFIG = {
    'primekg':  {'title': 'PrimeKG',                     'label_col': 'source',   'top_n': None},
    'drkg':     {'title': 'DRKG',                        'label_col': 'source',   'top_n': None},
    'hetionet': {'title': 'Hetionet',                    'label_col': 'metaedge', 'top_n': None},
    'openbilink':{'title': 'OpenBioLink',                 'label_col': 'source',   'top_n': None},
    'biokg':    {'title': 'BioKG',                       'label_col': 'predicate','top_n': None},
    'matrix':   {'title': 'MATRIX',                      'label_col': 'source',   'top_n': 20},
}

panel_kgs = [n for n in PANEL_CONFIG if n in kg_counts]
fig, axes  = plt.subplots(1, len(panel_kgs), figsize=(7.2, 3.6))
if len(panel_kgs) == 1:
    axes = [axes]

for ax, (i, name) in zip(axes, enumerate(panel_kgs)):
    pcfg   = PANEL_CONFIG[name]
    counts = kg_counts[name]
    top_n  = pcfg['top_n']
    d      = (counts.head(top_n) if top_n else counts).sort_values('n_edges')
    col    = pcfg['label_col']

    bars = ax.barh(d[col], d['n_edges'], color=KG_COLORS[name],
                   edgecolor='white', lw=0.5)
    for bar, v in zip(bars, d['n_edges']):
        ax.text(bar.get_width() + d['n_edges'].max() * 0.01,
                bar.get_y() + bar.get_height() / 2,
                f'{v:,}', va='center', fontsize=7.5, color=TEXT_COLOR)
    clean_ax(ax, title=pcfg['title'], xlabel='Edge count', grid_axis='x')
    ax.set_xlim(0, d['n_edges'].max() * 1.18)
    ax.tick_params(axis='y', labelsize=7.5)
    panel_label(ax, chr(ord('a') + i))

plt.suptitle('Source Database Diversity across Knowledge Graphs',
             fontsize=10, fontweight='bold', y=1.02, color=TEXT_COLOR)
plt.tight_layout()
save_fig(fig, FIGS, '03_source_diversity')
plt.show()

  → Saved: 03_source_diversity.pdf / .png


## 2. Edge Traceability

Source diversity tells us *which* databases contributed; edge traceability tells us *how finely* each individual edge can be traced back to its origin. We distinguish between **node-level** provenance (knowing which database an entity identifier comes from) and **edge-level** provenance (knowing which database asserted the relationship). We define three tiers of attribution precision:

| Tier | Description | Example | Implication |
|------|-------------|---------|-------------|
| **0, Type-level** | Source is implicit in edge/relation type; no per-edge or per-node field | Hetionet: 29 upstream databases, but no per-edge attribution in v1.0 | Cannot distinguish provenance of individual edges within the same type |
| **1, Per-node provenance** | Each endpoint is attributed to a source database, but the *edge itself* is not attributed | PrimeKG: `x_source`/`y_source` record which DB each entity comes from (e.g., DrugBank, NCBI) | Tells you where each node was sourced, but not who asserted the relationship |
| **2, Per-edge provenance** | Every edge is attributed to the database that asserted the relationship | DRKG: relation prefix (`DRUGBANK::ddi-interactor-in::...`) identifies the asserting source | Full edge-level traceability; can audit any edge back to its origin |

> **Important distinction.** PrimeKG's `x_source` / `y_source` fields record *node-level* provenance, e.g., a drug node sourced from DrugBank and a gene node from NCBI. This tells you *where the entity identifiers come from*, not *which database curated the interaction between them*. DRKG, by contrast, encodes the asserting source database directly in the relation name, providing genuine *edge-level* provenance.

Three empirical quantities are computed: (1) the percentage of edges carrying any source attribution, (2) the number of distinct contributing databases, and (3) for KGs with per-edge fields, the proportion of edges that draw from more than one source database.

In [6]:
# Source Granularity Index
GRANULARITY_CONFIG = {
    'primekg': {
        'tier':      1,
        'tier_label': '1 - per-node provenance',
        'mechanism': 'Explicit x_source / y_source columns (node-level: records which database each endpoint comes from, not which database asserted the edge)',
        'mode':      'columns',
        'cols':      ('x_source', 'y_source'),
        'documented_sources': 20,  # 20 primary resources per Chandak et al. 2023
    },
    'drkg': {
        'tier':      2,
        'tier_label': '2 - per-edge provenance',
        'mechanism': 'Source prefix in relation name (SRC::rel::Types) — each edge attributed to the database that asserted the relationship',
        'mode':      'relation_prefix',
        'documented_sources': 7,  # 7 primary resources per Ioannidis et al. 2020
    },
    'hetionet': {
        'tier':      0,
        'tier_label': '0 - type-level',
        'mechanism': None,  # built dynamically from metaedge count below
        'mode':      'relation_values',
        'documented_sources': 29,  # 29 primary resources per Himmelstein et al. 2017
    },
    # OpenBioLink x_source / y_source record the asserting database for every edge
    # (e.g. STRING, STITCH, Bgee, DisGeNet, SIDER, DrugCentral, GO, HPO, UBERON, DO, CDT).
    # Unlike PrimeKG (tier 1), these fields reflect *which database asserted the relationship*
    # true per-edge provenance , not merely which database the node identifier came from.
    'openbilink': {
        'tier':      2,
        'tier_label': '2 - per-edge provenance',
        'mechanism': 'Explicit x_source / y_source columns encoding the asserting database for each edge (STRING, STITCH, Bgee, DisGeNet, SIDER, DrugCentral, GO, HPO, UBERON, DO, CDT)',
        'mode':      'columns',
        'cols':      ('x_source', 'y_source'),
        'documented_sources': 11,  # 11 primary sources observed in edge data
    },
    # MATRIX exposes BOTH per-node provenance (x_source / y_source = which
    # Translator pipeline ingested each endpoint) AND per-edge provenance
    # (edge_source = the asserting database, e.g. inforesdrugbank). Score
    # against per-edge granularity (the more informative signal); the
    # node-level info is exposed separately for downstream auditing.
    'matrix': {
        'tier':      2,
        'tier_label':'2 - per-edge provenance (also per-node)',
        'mechanism': 'edge_source = primary_knowledge_source (Biolink infores: CURIE) per edge — 94 distinct asserting databases. Plus per-node x_source / y_source (upstream pipeline) and edge-level aggregator_source / edge_upstream — multi-tier provenance.',
        'mode':      'columns',
        'cols':      ('edge_source',),
        'documented_sources': 94,
    },
    'biokg': {
        'tier':      0,
        'tier_label': '0 - type-level',
        'mechanism': None,  # built dynamically from predicate count below
        'mode':      'relation_values',
        'documented_sources': 13,  # 13 primary databases per Walsh et al. 2020 (UniProt, DrugBank, KEGG, SIDER, HPA, Cellosaurus, Reactome, CTD, IntAct, MedGen, MESH, InterPro, SMPDB)
    },
}

# Tier → traceability score mapping (ordinal → 0-1)
# Tier 0 type-level only → 0.33 (no per-edge or per-node provenance)
# Tier 1 per-node provenance → 0.67 (know where endpoints came from, not who asserted the edge)
# Tier 2 per-edge provenance → 1.00 (every edge traceable to the database that asserted it)
TIER_SCORES = {0: 0.33, 1: 0.67, 2: 1.0}

def _granularity_row(name: str, kg: pd.DataFrame, cfg: dict) -> dict:
    mode = cfg['mode']
    n    = len(kg)

    if mode == 'columns':
        cols = cfg['cols']
        if len(cols) == 1:
            x = cols[0]
            attributed = kg[x].notna() & kg[x].astype(str).ne('')
            pct_attr   = 100 * attributed.sum() / n
            pct_cross  = 'N/A'  # single-col mode (edge-level); no x vs y comparison
        else:
            x, y       = cols
            attributed = kg[x].notna()
            pct_attr   = 100 * attributed.sum() / n
            cross      = attributed & kg[y].notna() & (kg[x] != kg[y])
            pct_cross  = 100 * cross.sum() / attributed.sum() if attributed.sum() else 0.0

    elif mode == 'relation_prefix':
        attributed = kg['relation'].str.contains('::')
        pct_attr   = 100 * attributed.sum() / n
        pct_cross  = 0.0  # single source per edge by design

    elif mode == 'relation_values':
        pct_attr  = 100.0  # every edge has a relation type by construction
        pct_cross = 'N/A'  # no per-edge source field

    # Use documented primary source count (from publication), not derived field count
    n_src = cfg['documented_sources']

    mechanism = cfg['mechanism'] or \
                f'Implicit via metaedge type ({kg["relation"].nunique()} types; no per-edge field)'

    return {
        'KG':                     name,
        'Granularity tier':       cfg['tier_label'],
        'Tier':                   cfg['tier'],
        'Mechanism':              mechanism,
        'Total edges':            n,
        'Edges attributed (%)':   round(pct_attr, 1),
        'Distinct source DBs':    n_src,
        'Cross-source edges (%)': round(pct_cross, 1) if isinstance(pct_cross, float) else pct_cross,
    }

gran_df = pd.DataFrame([
    _granularity_row(name, kgs[name]['kg'], cfg)
    for name, cfg in GRANULARITY_CONFIG.items()
    if name in kgs
]).set_index('KG')

gran_df

,Granularity tier,Tier,Mechanism,Total edges,Edges attributed (%),Distinct source DBs,Cross-source edges (%)
KG,,,,,,,
primekg,1 - per-node provenance,1,Explicit x_source / y_source columns (node-lev...,8100498,100.0,20,55.8
drkg,2 - per-edge provenance,2,Source prefix in relation name (SRC::rel::Type...,5874261,100.0,7,0.0
hetionet,0 - type-level,0,Implicit via metaedge type (24 types; no per-e...,2250197,100.0,29,N/A
openbilink,2 - per-edge provenance,2,Explicit x_source / y_source columns encoding ...,4986521,100.0,11,0.0
matrix,2 - per-edge provenance (also per-node),2,edge_source = primary_knowledge_source (Biolin...,41595079,100.0,94,N/A
biokg,0 - type-level,0,Implicit via metaedge type (17 types; no per-e...,2067998,100.0,13,N/A


### Diagnostic Edge Provenance Examples

The following samples illustrate the concrete provenance information available at the edge level in each KG. This helps developers understand *exactly* what metadata is (or is not) attached to individual edges, and where to look when auditing a specific claim.

In [7]:
# Edge provenance examples , show what metadata each KG attaches to individual edges
import random as _rng
_rng.seed(params.get('random_seed', 42))
N_EDGE_EXAMPLES = 5

for name in KG_NAMES:
    kg = kgs[name]['kg']
    print(f'\n{"=" * 80}')
    print(f'{name.upper()} — edge-level provenance ({len(kg):,} edges, {len(kg.columns)} columns)')
    print(f'{"=" * 80}')

    # Show available columns
    provenance_cols = [c for c in kg.columns
                       if any(kw in c.lower() for kw in
                              ('source', 'evidence', 'pubmed', 'pmid', 'doi',
                               'score', 'confidence', 'weight', 'prob'))]
    print(f'\nProvenance-related columns: {provenance_cols if provenance_cols else "NONE"}')
    print(f'All columns: {list(kg.columns)}\n')

    # Sample edges and display with all columns
    sample_idx = _rng.sample(range(len(kg)), min(N_EDGE_EXAMPLES, len(kg)))
    sample = kg.iloc[sample_idx]

    # For readability, show only non-index columns with actual content
    show_cols = [c for c in kg.columns if c not in ('x_index', 'y_index')
                 and sample[c].notna().any()]
    display(sample[show_cols])

    # Highlight provenance implications
    cfg = GRANULARITY_CONFIG.get(name, {})
    tier = cfg.get('tier', -1)
    if tier == 0:
        print(f'\n→ Tier 0: No per-edge source field. To audit an edge, you must consult')
        print(f'  the original publication to determine which of the {cfg.get("documented_sources", "?")} upstream')
        print(f'  databases contributed it, using the relation type as the only discriminator.')
    elif tier == 1:
        cols = cfg.get('cols', ())
        print(f'\n→ Tier 1: Node-level provenance via {cols}.')
        print(f'  You can see which database each endpoint comes from, but NOT which')
        print(f'  database asserted the relationship between them.')
        # Show cross-source edge example
        if len(cols) == 2:
            cross = kg[kg[cols[0]].notna() & kg[cols[1]].notna() & (kg[cols[0]] != kg[cols[1]])]
            if not cross.empty:
                ex = cross.iloc[0]
                print(f'  Example cross-source edge: {ex[cols[0]]} node ↔ {ex[cols[1]]} node')
    elif tier == 2:
        print(f'\n→ Tier 2: Per-edge provenance encoded in relation name prefix.')
        print(f'  Every edge is fully traceable to the database that asserted it.')
        # Show distinct source prefixes
        prefixes = kg['relation'].str.extract(r'^([^:]+)::')[0].dropna().unique()
        print(f'  Source prefixes: {sorted(prefixes)}')


PRIMEKG — edge-level provenance (8,100,498 edges, 12 columns)

Provenance-related columns: ['x_source', 'y_source']
All columns: ['relation', 'display_relation', 'x_index', 'x_id', 'x_type', 'x_name', 'x_source', 'y_index', 'y_id', 'y_type', 'y_name', 'y_source']



,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
5363900,anatomy_protein_present,expression present,102288414,gene/protein,C11orf98,NCBI,1225,anatomy,cortex of kidney,UBERON
933912,drug_drug,synergistic interaction,DB09111,drug,Pentastarch,DrugBank,DB00438,drug,Ceftazidime,DrugBank
209805,protein_protein,ppi,4815,gene/protein,NINJ2,NCBI,120939,gene/protein,TMEM52B,NCBI
6220576,molfunc_protein,interacts with,42803,molecular_function,protein homodimerization activity,GO,3087,gene/protein,HHEX,NCBI
2307113,drug_drug,synergistic interaction,DB04930,drug,Permethrin,DrugBank,DB12783,drug,Benserazide,DrugBank



→ Tier 1: Node-level provenance via ('x_source', 'y_source').
  You can see which database each endpoint comes from, but NOT which
  database asserted the relationship between them.


  Example cross-source edge: DrugBank node ↔ NCBI node

HETIONET — edge-level provenance (2,250,197 edges, 9 columns)

Provenance-related columns: NONE
All columns: ['relation', 'x_index', 'x_id', 'y_index', 'y_id', 'x_type', 'x_name', 'y_type', 'y_name']



,relation,x_id,y_id,x_type,x_name,y_type,y_name
1027150,GcG,Gene::10499,Gene::7545,Gene,NCOA2,Gene,ZIC1
936213,AuG,Anatomy::UBERON:0002037,Gene::2534,Anatomy,cerebellum,Gene,FYN
585264,GiG,Gene::23741,Gene::121504,Gene,EID1,Gene,HIST4H4
429895,GpBP,Gene::10978,Biological Process::GO:0006399,Gene,CLP1,Biological Process,tRNA metabolic process
364647,GpBP,Gene::6921,Biological Process::GO:0043902,Gene,TCEB1,Biological Process,positive regulation of multi-organism process


...

,relation,x_id,x_type,x_name,y_id,y_type,y_name
416992,MEMBER_OF_COMPLEX,Q71U36,Gene/Protein,TBA1A,R-HSA-1638796,Complex,R-HSA-1638796
1501601,DDI,DB06695,Drug,Dabigatran_etexilate,DB09241,Drug,Methylene_blue
1362906,DDI,DB00238,Drug,Nevirapine,DB13953,Drug,Estradiol_benzoate
1470785,DDI,DB00012,Drug,Darbepoetin_alfa,DB13066,Drug,Liarozole
1142825,DDI,DB00252,Drug,Phenytoin,DB06272,Drug,Maxacalcitol



→ Tier 0: No per-edge source field. To audit an edge, you must consult
  the original publication to determine which of the 13 upstream
  databases contributed it, using the relation type as the only discriminator.

MATRIX — edge-level provenance (41,595,079 edges, 18 columns)

Provenance-related columns: ['x_source', 'y_source', 'edge_source', 'aggregator_source', 'confidence_score']
All columns: ['relation', 'x_index', 'x_id', 'x_type', 'x_name', 'x_source', 'y_index', 'y_id', 'y_type', 'y_name', 'y_source', 'edge_source', 'aggregator_source', 'edge_upstream', 'knowledge_level', 'confidence_score', 'agent_type', 'num_references']



,relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source,edge_source,aggregator_source,edge_upstream,knowledge_level,confidence_score,agent_type
28153498,related_to,PUBCHEM.COMPOUND:131760734,drug,"TG(18:1(9Z)/22:1(13Z)/22:4(7Z,10Z,13Z,16Z))",rtxkg2|robokop,335,gene/protein,APOA1,primekg|rtxkg2|robokop,infores:hmdb,['infores:rtx-kg2'],['rtxkg2'],knowledge_assertion,1.0,manual_agent
14793519,disrupts,UMLS:C0540142,gene/protein,methionine aminopeptidase 2,rtxkg2,MONDO:0004992,effect/phenotype,Cancer,primekg|rtxkg2|robokop,infores:semmeddb,['infores:rtx-kg2'],['rtxkg2'],prediction,0.3,text_mining_agent
30145908,coexpressed_with,51412,gene/protein,"ACTL6B protein, human",primekg|rtxkg2|robokop,7404,gene/protein,"UTY protein, human",primekg|rtxkg2|robokop,infores:string,['infores:robokop-kg'],['robokop'],knowledge_assertion,1.0,manual_agent
39544950,coexpressed_with,80148,gene/protein,solute carrier family 66 member 2 isoform 1 (h...,rtxkg2|robokop,131583,gene/protein,FAM43A,primekg|rtxkg2|robokop,infores:string,['infores:robokop-kg'],['robokop'],knowledge_assertion,1.0,manual_agent
18669062,coexpressed_with,9019,gene/protein,myelin protein zero-like protein 1 isoform h1 ...,rtxkg2|robokop,1718,gene/protein,Delta(24)-sterol reductase isoform 1 (human),rtxkg2|robokop,infores:string,['infores:robokop-kg'],['robokop'],knowledge_assertion,1.0,manual_agent



→ Tier 2: Per-edge provenance encoded in relation name prefix.
  Every edge is fully traceable to the database that asserted it.


  Source prefixes: []


## 3. Uncertainty Quantification

A trustworthy KG should not only record what is known, but also how confidently it is known. We assess whether each KG encodes per-edge confidence scores or uncertainty estimates by scanning for columns whose names contain score-related keywords (`score`, `confidence`, `weight`, `prob`, `evidence`, `support`).

The **uncertainty quantification** sub-metric is the fraction of edges that carry any numeric confidence or evidence-strength annotation:

$$\text{Uncertainty Quantification}_i = \frac{\text{edges with confidence score}_i}{\text{total edges}_i}$$

This metric ranges from 0 (no edges scored) to 1 (every edge has a confidence value). Currently, none of the benchmark KGs provide per-edge confidence scores, yielding 0.0 for all, a finding that motivates future KG design to include quantified evidence strength.

In [8]:
# Uncertainty Quantification confidence score field availability
_SCORE_KEYWORDS = {'score', 'confidence', 'weight', 'prob', 'evidence', 'support'}

def _score_rows(name: str, kg: pd.DataFrame) -> List[dict]:
    score_cols = [c for c in kg.columns
                  if any(kw in c.lower() for kw in _SCORE_KEYWORDS)]
    if not score_cols:
        return [{'KG': name, 'Score column': '-- none found --',
                 'Edges with score (%)': 0.0, 'Score range': 'N/A'}]
    rows = []
    for col in score_cols:
        non_null = kg[col].notna().sum()
        pct      = 100 * non_null / len(kg)
        rows.append({
            'KG':                   name,
            'Score column':         col,
            'Edges with score (%)': round(pct, 1),
            'Score range':          f'{kg[col].min():.3g} to {kg[col].max():.3g}'
                                    if pct > 0 else 'N/A',
        })
    return rows

uq_df = pd.DataFrame([
    row
    for name in KG_NAMES
    for row in _score_rows(name, kgs[name]['kg'])
]).set_index('KG')

print('Uncertainty / confidence score availability:')
display(uq_df)

# Compute UQ coverage score fraction of edges with any confidence annotation
uq_coverage = {}
for name in KG_NAMES:
    kg = kgs[name]['kg']
    score_cols = [c for c in kg.columns
                  if any(kw in c.lower() for kw in _SCORE_KEYWORDS)]
    if not score_cols:
        uq_coverage[name] = 0.0
    else:
        # Union of edges that have at least one score column populated
        has_any = kg[score_cols].notna().any(axis=1)
        uq_coverage[name] = round(has_any.sum() / len(kg), 4)

print('\nUQ coverage scores (fraction of edges with confidence annotation):')
for kg_name, s in uq_coverage.items():
    print(f'  {kg_name:12s}: {s:.4f}')

Uncertainty / confidence score availability:


,Score column,Edges with score (%),Score range
KG,,,
primekg,-- none found --,0.0,N/A
hetionet,-- none found --,0.0,N/A
drkg,-- none found --,0.0,N/A
openbilink,-- none found --,0.0,N/A
biokg,-- none found --,0.0,N/A
matrix,confidence_score,93.4,0.3 to 1



UQ coverage scores (fraction of edges with confidence annotation):
  primekg     : 0.0000
  hetionet    : 0.0000
  drkg        : 0.0000
  openbilink  : 0.0000
  biokg       : 0.0000
  matrix      : 0.9337


In [9]:
# Biolink knowledge_level breakdown , categorical uncertainty signal
# (currently only MATRIX exposes this column; the loop is generic so it
# automatically extends to any future KG that propagates it.)
_KL_ORDER = ['knowledge_assertion', 'logical_entailment', 'statistical_association',
             'observation', 'prediction', 'not_provided']

kl_rows = []
for name in KG_NAMES:
    kg = kgs[name]['kg']
    if 'knowledge_level' not in kg.columns:
        continue
    counts = kg['knowledge_level'].fillna('not_provided').value_counts()
    total  = int(counts.sum())
    for lvl in _KL_ORDER:
        n = int(counts.get(lvl, 0))
        kl_rows.append({
            'KG':                 name,
            'knowledge_level':    lvl,
            'edges':              n,
            'pct':                round(100 * n / total, 2) if total else 0.0,
        })

if kl_rows:
    kl_df = pd.DataFrame(kl_rows)
    print('Biolink knowledge_level distribution:')
    pivot = (kl_df.pivot(index='knowledge_level', columns='KG', values='pct')
             .reindex(_KL_ORDER).fillna(0.0))
    display(pivot)

    # Headline metric fraction of edges with high-trust assertion
    print('\nHigh-trust fraction (knowledge_assertion + logical_entailment):')
    for name in pivot.columns:
        high = pivot.loc[['knowledge_assertion','logical_entailment'], name].sum()
        print(f'  {name:12s}: {high:5.1f}%')
else:
    print('No KG with knowledge_level column found — skipping.')


Biolink knowledge_level distribution:


KG,matrix
knowledge_level,
knowledge_assertion,65.36
logical_entailment,0.61
statistical_association,0.01
observation,0.00
prediction,27.39
not_provided,6.63



High-trust fraction (knowledge_assertion + logical_entailment):
  matrix      :  66.0%


In [10]:
# Checkpoint
CKPT_DIR = BASE / 'results' / 'checkpoints'
CKPT_DIR.mkdir(exist_ok=True)

# 1. Edge traceability score (tier → 0-1, equal-spaced ordinal)
# Tier 0 type-level only → 0.00 (no per-edge or per-node provenance)
# Tier 1 per-node provenance → 0.50 (know where endpoints came from, not who asserted the edge)
# Tier 2 per-edge provenance → 1.00 (every edge traceable to the database that asserted it)
TIER_SCORES = {0: 0.0, 1: 0.5, 2: 1.0}

traceability_scores = {}
for kg_name in KG_NAMES:
    if kg_name in gran_df.index:
        tier = int(gran_df.loc[kg_name, 'Tier'])
        traceability_scores[kg_name] = TIER_SCORES[tier]
    else:
        traceability_scores[kg_name] = 0.0

print('Edge traceability scores:')
for kg, s in traceability_scores.items():
    tier = gran_df.loc[kg, 'Granularity tier'] if kg in gran_df.index else 'N/A'
    print(f'  {kg:12s}: {s:.2f}  (tier: {tier})')

# 2. Source diversity (log-normalised 0-1 score)
# Score = log(n_sources) / log(max_sources) across the benchmark KGs.
# Log transform reflects the diminishing marginal value of additional
# sources (Haynes et al. 2018) the information gain from adding a 5th
# source is larger than adding a 30th.
import math

source_counts = {}
for kg_name in KG_NAMES:
    if kg_name in gran_df.index:
        source_counts[kg_name] = int(gran_df.loc[kg_name, 'Distinct source DBs'])
    else:
        source_counts[kg_name] = 0

max_sources = max(source_counts.values())
source_diversity_scores = {}
for kg_name, n in source_counts.items():
    if n > 0 and max_sources > 1:
        source_diversity_scores[kg_name] = round(math.log(n) / math.log(max_sources), 4)
    else:
        source_diversity_scores[kg_name] = 0.0

print('\nSource diversity scores (log-normalised, max = ' + str(max_sources) + ' sources):')
for kg, s in source_diversity_scores.items():
    print(f'  {kg:12s}: {s:.4f}  ({source_counts[kg]} sources)')

# 3. UQ coverage score (% edges with confidence annotation)
print('\nUQ coverage scores:')
for kg, s in uq_coverage.items():
    print(f'  {kg:12s}: {s:.4f}')

# 4. Summary = equal-weighted mean of traceability + source diversity + UQ
sub_scores = {}
summary_scores = {}
for kg_name in KG_NAMES:
    t = traceability_scores[kg_name]
    d = source_diversity_scores[kg_name]
    u = uq_coverage[kg_name]
    sub_scores[kg_name] = {
        'edge_traceability':     t,
        'source_diversity':      d,
        'source_diversity_raw':  source_counts[kg_name],
        'uq_coverage':           u,
    }
    summary_scores[kg_name] = round((t + d + u) / 3.0, 4)

print(f'\nTrustworthiness (3 scored sub-metrics, equal-weighted):')
for kg_name in KG_NAMES:
    ss = sub_scores[kg_name]
    print(f'  {kg_name:12s}: {summary_scores[kg_name]:.4f}  '
          f'(traceability={ss["edge_traceability"]:.2f}, '
          f'source_div={ss["source_diversity"]:.4f}, '
          f'uq={ss["uq_coverage"]:.4f})')

# 5. Save checkpoint
ckpt = {
    'summary_scores': summary_scores,
    'sub_scores':     sub_scores,
    'gran_records':   gran_df.reset_index().to_dict('records') if not gran_df.empty else [],
}
out = CKPT_DIR / '03_trustworthiness.pkl'
with open(out, 'wb') as f:
    pickle.dump(ckpt, f)
print(f'\nCheckpoint saved: {out}')

Edge traceability scores:
  primekg     : 0.50  (tier: 1 - per-node provenance)
  hetionet    : 0.00  (tier: 0 - type-level)
  drkg        : 1.00  (tier: 2 - per-edge provenance)
  openbilink  : 1.00  (tier: 2 - per-edge provenance)
  biokg       : 0.00  (tier: 0 - type-level)
  matrix      : 1.00  (tier: 2 - per-edge provenance (also per-node))

Source diversity scores (log-normalised, max = 94 sources):
  primekg     : 0.6594  (20 sources)
  hetionet    : 0.7412  (29 sources)
  drkg        : 0.4283  (7 sources)
  openbilink  : 0.5278  (11 sources)
  biokg       : 0.5646  (13 sources)
  matrix      : 1.0000  (94 sources)

UQ coverage scores:
  primekg     : 0.0000
  hetionet    : 0.0000
  drkg        : 0.0000
  openbilink  : 0.0000
  biokg       : 0.0000
  matrix      : 0.9337

Trustworthiness (3 scored sub-metrics, equal-weighted):
  primekg     : 0.3865  (traceability=0.50, source_div=0.6594, uq=0.0000)
  hetionet    : 0.2471  (traceability=0.00, source_div=0.7412, uq=0.0000)
  drkg

## Dimension score

The trustworthiness dimension score combines three scored sub-metrics:

$$\text{Trustworthiness} = \frac{1}{3}\left(\text{Edge Traceability} + \text{Source Diversity} + \text{Uncertainty Quantification}\right)$$

**Edge traceability** maps the ordinal granularity tier to a 0-1 score using equal spacing across three levels:
- Tier 0 (type-level only) → 0.00
- Tier 1 (per-node provenance) → 0.50
- Tier 2 (per-edge provenance) → 1.00

The key distinction is between *node-level* provenance (which database an entity ID comes from) and *edge-level* provenance (which database asserted the relationship). Edge-level attribution receives the highest score because it enables auditing individual claims.

**Source diversity** is log-normalised against the maximum observed source count across the benchmark KGs:

$$\text{Source Diversity}_i = \frac{\log(n_i)}{\log(n_{\max})}$$

The log transform reflects the diminishing marginal information gain from additional sources (Haynes et al. 2018): the information gain from adding a 5th independent database is substantially larger than adding a 30th. The normalisation ceiling is the maximum observed count across the benchmark KGs (currently Hetionet with 29 sources), ensuring scores remain in [0, 1] and are interpretable as a fraction of the benchmark maximum.

**Uncertainty quantification** is the fraction of edges carrying any numeric confidence or evidence-strength annotation:

$$\text{Uncertainty Quantification}_i = \frac{\text{edges with confidence score}_i}{\text{total edges}_i}$$

Currently 0.0 for all benchmark KGs, reflecting a systematic absence of per-edge confidence scores in published biomedical KGs.

In [11]:
# Memory cleanup - free state before the next notebook in the pipeline.
# Without this, the kgs / graphs / maps dicts (multi-GB pandas DataFrames
# and NetworkX graphs) stay resident in the kernel; the next notebook then
# re-loads everything on top, which is how we pushed past 150 GB.
_to_free = ['kgs', 'graphs', 'maps', 'preps']
for _v in _to_free:
    try:
        del globals()[_v]
    except KeyError:
        pass
import gc
gc.collect()
print('Freed KG state from kernel memory.')

Freed KG state from kernel memory.
